In [1]:
import pandas as pd

data = pd.read_csv("../../data/raw/kathmandu_full_raw_2023_2024.csv")

Datetime format conversion and sorting (although already sorted)

In [2]:

data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)

Create target column. For instance i, target is PM2.5 of instance (i+1) i.e. PM2.5 after 1 hour.

In [3]:
data["target"] = data["pm2_5"].shift(-1)
data.head(3)

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,target
0,2023-01-01 00:00:00,83.4,119.2,1689,38.0,12.9,67,7.9,88,1.1,162,872.6,82.2
1,2023-01-01 01:00:00,82.2,117.6,1594,31.5,13.2,70,8.0,85,2.9,150,872.3,80.8
2,2023-01-01 02:00:00,80.8,115.7,1504,24.6,13.6,73,8.0,81,4.1,135,871.9,78.2


PM2.5 lag features. i.e. PM2.5 at (t-1), (t-2), ... (t-12)

In [4]:
for i in range(1, 13):
    data[f"pm2_5_lag_{i}"] = data["pm2_5"].shift(i)

data[20:25]

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,wind_speed_10m,...,pm2_5_lag_3,pm2_5_lag_4,pm2_5_lag_5,pm2_5_lag_6,pm2_5_lag_7,pm2_5_lag_8,pm2_5_lag_9,pm2_5_lag_10,pm2_5_lag_11,pm2_5_lag_12
20,2023-01-01 20:00:00,88.2,127.1,1713,52.2,11.1,62,9.2,81,3.1,...,66.8,63.8,62.4,62.2,64.0,69.2,69.8,70.3,71.5,88.5
21,2023-01-01 21:00:00,95.2,136.9,1845,54.2,11.4,56,8.3,85,3.6,...,75.0,66.8,63.8,62.4,62.2,64.0,69.2,69.8,70.3,71.5
22,2023-01-01 22:00:00,96.6,138.8,1921,53.5,11.8,52,7.4,89,2.0,...,81.0,75.0,66.8,63.8,62.4,62.2,64.0,69.2,69.8,70.3
23,2023-01-01 23:00:00,93.1,134.1,1918,48.8,12.1,53,6.7,92,2.0,...,88.2,81.0,75.0,66.8,63.8,62.4,62.2,64.0,69.2,69.8
24,2023-01-02 00:00:00,90.3,129.9,1795,41.6,12.6,59,6.0,93,2.5,...,95.2,88.2,81.0,75.0,66.8,63.8,62.4,62.2,64.0,69.2


Pollutant lag features, for pollutants, (t-1), (t-2), (t-3).

In [5]:
pollutants = ["pm10", "carbon_monoxide", "nitrogen_dioxide", "sulphur_dioxide", "ozone"]

for col in pollutants:
    for i in range(1, 4):
        data[f"{col}_lag_{i}"] = data[col].shift(i)

Meterological lag features, (t-1).

In [6]:
meteo = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_direction_10m", "surface_pressure"]

for col in meteo:
    data[f"{col}_lag_1"] = data[col].shift(1)

Handling null values

In [7]:
data = data.dropna().reset_index(drop=True)

In [8]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 17531 entries, 0 to 17530
Data columns (total 45 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   time                        17531 non-null  datetime64[us]
 1   pm2_5                       17531 non-null  float64       
 2   pm10                        17531 non-null  float64       
 3   carbon_monoxide             17531 non-null  int64         
 4   nitrogen_dioxide            17531 non-null  float64       
 5   sulphur_dioxide             17531 non-null  float64       
 6   ozone                       17531 non-null  int64         
 7   temperature_2m              17531 non-null  float64       
 8   relative_humidity_2m        17531 non-null  int64         
 9   wind_speed_10m              17531 non-null  float64       
 10  wind_direction_10m          17531 non-null  int64         
 11  surface_pressure            17531 non-null  float64       
 12  t

These are all our numeric columns up until now.  

In [9]:
numerical_cols = list(data.drop(columns=["time", "target"]).columns)

Now, we work towards our categorical features.

Meterological bin features. (based on meterological analysis in EDA)

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class MeterologicalBinFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["temp_bin"] = pd.cut(
            X["temperature_2m"],
            bins=[-np.inf, 10, 20, np.inf],
            labels=["low", "medium", "high"]
        )

        X["humidity_bin"] = pd.cut(
            X["relative_humidity_2m"],
            bins=[-np.inf, 40, np.inf],
            labels=["low", "normal"]
        )

        X["wind_speed_bin"] = pd.cut(
            X["wind_speed_10m"],
            bins=[-np.inf, 3, 6, 12, 15, np.inf],
            labels=["calm", "light", "moderate", "strong", "very_strong"]
        )

        X["wind_direction_bin"] = np.where(
            (X["wind_direction_10m"] > 120) & (X["wind_direction_10m"] <= 240),
            "mid",
            "outer"
        )

        X["surface_pressure_bin"] = pd.cut(
            X["surface_pressure"],
            bins=[-np.inf, 865, 870, 875, np.inf],
            labels=["very_low", "low", "normal", "high"]
        )

        return X

Temporal bin features. (based on temporal analysis in EDA)

In [11]:
class TemporalBinFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["hour"] = X["time"].dt.hour
        X["month"] = X["time"].dt.month
        X["day_of_week"] = X["time"].dt.dayofweek


        X["season"] = np.where(
            X["month"].isin([11, 12, 1, 2]),
            "Nov_Feb",
            np.where(
                X["month"].isin([3, 4, 5, 6]),
                "Mar_Jun",
                "Jul_Oct"
            )
        )

        X["time_of_day"] = np.where(
            X["hour"].isin([19, 20, 21, 22, 23, 0]),
            "7pm_to_12am",
            np.where(
                X["hour"].isin([1, 2, 3, 4, 5, 6, 7, 8]),
                "1am_to_8am",
                np.where(
                    X["hour"].isin([9, 10, 11, 12, 13, 14, 15, 16]),
                    "9am_to_4pm",
                    "5pm_to_6pm"
                )
            )
        )
                
        return X


In [12]:
categorical_cols = [
    "temp_bin",
    "humidity_bin",
    "wind_speed_bin",
    "wind_direction_bin",
    "surface_pressure_bin",
    "season",
    "time_of_day",
]

Cycle encoding hour, day and month (raw numeric values can negatively affect some models)

In [13]:
class CyclicalFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # cyclical features
        X["hour_sin"] = np.sin(2*np.pi*X["hour"]/24)
        X["hour_cos"] = np.cos(2*np.pi*X["hour"]/24)

        X["dow_sin"] = np.sin(2*np.pi*X["day_of_week"]/7)
        X["dow_cos"] = np.cos(2*np.pi*X["day_of_week"]/7)

        X["month_sin"] = np.sin(2*np.pi*X["month"]/12)
        X["month_cos"] = np.cos(2*np.pi*X["month"]/12)

        X = X.drop(columns=["hour", "day_of_week", "month"])

        return X

In [14]:
numerical_cols = numerical_cols + ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"]

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("meterological", MeterologicalBinFeatures()),
    ("temporal", TemporalBinFeatures()),
    ("cyclical", CyclicalFeatures()),
    ("preprocess", ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("scaler", StandardScaler(), numerical_cols)
        ]
    ))
])


Now lets process our data and test.

In [16]:
X = data.drop(columns=["target"])
y = data["target"]

split = int(np.ceil(0.8 * len(X)))

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [17]:
X_train_t = pipeline.fit_transform(X_train)
X_test_t = pipeline.transform(X_test)

In [18]:
from sklearn.linear_model import LinearRegression

linear_reg = LinearRegression()
linear_reg.fit(X_train_t, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [19]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

y_pred = linear_reg.predict(X_test_t)
print(root_mean_squared_error(y_pred, y_test))
print(mean_absolute_error(y_pred, y_test))

6.774044591838469
3.702794440599946


In [34]:
from sklearn.ensemble import RandomForestRegressor

random_forest = RandomForestRegressor(
    n_estimators=20,
    max_depth=15,              
    min_samples_split=5,       
    min_samples_leaf=2,        
    max_features='sqrt',    
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train_t, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",20
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [35]:
y_pred = random_forest.predict(X_test_t)
print(root_mean_squared_error(y_pred, y_test))
print(mean_absolute_error(y_pred, y_test))

13.325455089857705
7.060984301677221
